In [ ]:
!pip install -U --upgrade-strategy eager pydantic langchain-openai

In [ ]:
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

In [ ]:
import os
import requests
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent

import utils
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

import numpy as np

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
@tool
def compute_simple_interest(principal: float, rate:float, time:float)-> str:
    """Takes principal, rate, time as input. Returns simple interest rounded to two decimal places"""
    si = np.round(principal*rate*time/100,2)
    return f"Computed simple interest is {si}"

@tool
def compute_compound_interest(principal: float, rate:float, time:float)-> str:
    """Takes principal, rate, time as input. Returns compound interest rounded to two decimal places"""
    ci = np.round(principal*np.power((1+rate/100),time),2)-principal
    return f"Computed compound interest is {ci}"

In [ ]:
SYSTEM_PROMPT = """You are an expert financial agent who can compute interest.
You should use the tool given to respond to the question.
Do not use any other knowledge for the response
"""

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system", SYSTEM_PROMPT
    ),
    (
        "human", "User: {input}"
    ),
    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    )
])

In [ ]:
tools = [compute_compound_interest,compute_simple_interest]

In [ ]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [ ]:
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True    
)

In [ ]:
response = executor.invoke({"input": "Compute Simple Interest on a principal of Rs 10000 for 2 years at 12 percent rate"})
print("\n--- Interest Computing Agent Output ---")
print(response["output"])

In [ ]:
response = executor.invoke({"input": "Compute compounded interest on a sum of Rs 40000 for 3 years at 10 percent rate"})
print("\n--- Interest Computing Agent Output ---")
print(response["output"])